# Oneiromantia — Runnable Codebase Notebook

This notebook materializes the Oneiromantia multi-agent dream-analysis pipeline on disk and runs it end-to-end. It's written to run unmodified on **Kaggle** (Save & Run All, for competition submission) or **Google Colab** — it detects which platform it's on and adapts secret lookup and file paths accordingly.

Pipeline: `symbol_extractor` -> `pattern_analyst` -> `art_generator`, orchestrated by an ADK `Workflow`, with a local MCP server (`dream_journal_mcp`) persisting symbol history between runs.

**To see the real analysis run:** provide a `GOOGLE_API_KEY`.
- Kaggle: Add-ons > Secrets > add `GOOGLE_API_KEY`, attach it to this notebook, and enable **Internet** (Settings panel).
- Colab: click the key icon in the left sidebar > add `GOOGLE_API_KEY` > toggle notebook access on. Internet is on by default.

Without a key, the notebook still runs end-to-end: Section 5 proves the pipeline is wired correctly (no API calls), and Section 6 (the live demo) reports that it's skipped instead of failing.

## 1. Install dependencies

In [1]:
%pip install -q "google-adk>=2.3.0" "mcp>=1.28.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 742.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Configure the Gemini API key

Tries Kaggle Secrets, then Colab's userdata secrets, then a plain environment variable — whichever is present. This does **not** hard-fail if no key is found. Section 5 below always runs a no-key wiring self-test (installs, file materialization, imports, and full agent-graph construction) so the notebook still demonstrates the pipeline is correctly assembled even without credentials. Section 6 (the live Gemini demo) is skipped gracefully if no key is found, or is rejected.

In [2]:
import os

GOOGLE_API_KEY = ""

try:
    from kaggle_secrets import UserSecretsClient
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
except Exception:
    pass

if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

HAS_KEY = bool(GOOGLE_API_KEY)

if HAS_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini API key configured \u2014 the live demo in Section 6 will run.")
else:
    print(
        "No GOOGLE_API_KEY found.\n"
        "  Kaggle: Add-ons > Secrets > add GOOGLE_API_KEY, then attach it to this notebook.\n"
        "  Colab:  key icon (left sidebar) > add GOOGLE_API_KEY, then enable notebook access.\n"
        "  Local:  export GOOGLE_API_KEY before launching Jupyter.\n"
        "Continuing without one \u2014 Section 5's wiring self-test doesn't need "
        "a key; Section 6's live demo will be skipped."
    )


Gemini API key configured — the live demo in Section 6 will run.


## 3. Materialize the project source on disk

The cells below use `%%writefile` to write each module to its real relative path (mirroring the source repository layout). This matters for two reasons: intra-project imports like `from symbol_extractor import make_symbol_extractor` need real importable packages, and `orchestrator/agent.py` locates the MCP server via `os.path.abspath(__file__)`, which only resolves correctly for code that actually lives on disk (not code pasted into a notebook cell).

`PROJECT_ROOT` uses Kaggle's writable `/kaggle/working` directory when present; otherwise it falls back to the current working directory, which is `/content` on Colab (and the notebook's own directory when run locally).

In [3]:
import os

PROJECT_ROOT = (
    "/kaggle/working/oneiromantia"
    if os.path.isdir("/kaggle/working")
    else os.path.join(os.getcwd(), "oneiromantia")
)
API_DIR = f"{PROJECT_ROOT}/apps/api"
PACKAGES_DIR = f"{PROJECT_ROOT}/packages"

for _d in [
    f"{API_DIR}/symbol_extractor",
    f"{API_DIR}/pattern_analyst",
    f"{API_DIR}/art_generator",
    f"{API_DIR}/orchestrator",
    f"{API_DIR}/dream_journal_mcp",
    f"{PACKAGES_DIR}/prompts",
]:
    os.makedirs(_d, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: /kaggle/working/oneiromantia


### `apps/api/main.py`

In [4]:
%%writefile {API_DIR}/main.py
import uuid
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

from symbol_extractor import make_symbol_extractor
from pattern_analyst import make_pattern_analyst
from art_generator import make_art_generator
from orchestrator import make_orchestrator

def build_oneiro_app() -> Runner:
    """
    Wire everything together and return a Runner ready for use.
    """
    symbol_extractor = make_symbol_extractor()
    pattern_analyst  = make_pattern_analyst()
    art_generator    = make_art_generator()
    orchestrator     = make_orchestrator(symbol_extractor, pattern_analyst, art_generator)

    session_service = InMemorySessionService()

    runner = Runner(
        agent=orchestrator,
        app_name="oneiro",
        session_service=session_service,
    )

    return runner


async def run_demo():
    """
    Quick demo: submit one dream entry and print the orchestrator's response.
    """
    runner = build_oneiro_app()

    user_id    = "demo-user"
    session_id = str(uuid.uuid4())

    # Create a session first
    await runner.session_service.create_session(
        app_name="oneiro",
        user_id=user_id,
        session_id=session_id,
    )

    dream_input = (
        "I was standing at the edge of a vast black ocean. "
        "The water was completely still, like a mirror. "
        "Behind me was a glass tower that went up beyond the clouds. "
        "I felt a strange mixture of awe and dread — like I was being watched "
        "by something in the reflection."
    )

    print(f"Session: {session_id}")
    print(f"Dream  : {dream_input[:60]}...\n")
    print("=" * 60)

    # Stream the response turn by turn
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=genai_types.Content(
            role="user",
            parts=[genai_types.Part(text=dream_input)],
        ),
    ):
        # Print agent activity as it streams
        if hasattr(event, "content") and event.content:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(part.text, end="", flush=True)

    print("\n" + "=" * 60)


if __name__ == "__main__":
    asyncio.run(run_demo())


Writing /kaggle/working/oneiromantia/apps/api/main.py


### `apps/api/symbol_extractor/`

In [5]:
%%writefile {API_DIR}/symbol_extractor/__init__.py
from .agent import make_symbol_extractor

__all__ = ["make_symbol_extractor"]


Writing /kaggle/working/oneiromantia/apps/api/symbol_extractor/__init__.py


In [6]:
%%writefile {API_DIR}/symbol_extractor/agent.py
from google.adk.agents.llm_agent import Agent
from pydantic import BaseModel, Field

class SymbolItem(BaseModel):
    name: str = Field(description="Name of the symbol, e.g. 'water', 'tower'")
    recurring: bool = Field(description="True if the symbol has appeared in previous dreams, False otherwise")

class SymbolExtractorOutput(BaseModel):
    symbols: list[SymbolItem] = Field(description="List of archetypal symbols extracted from the dream")
    emotions: list[str] = Field(description="Dominant emotional tones felt in the dream")
    setting: str = Field(description="Primary setting or environment")

def make_symbol_extractor() -> Agent:
    """
    Sub-agent 1: Symbol Extractor
    Input  : normalized dream text + user symbol history (from MCP context)
    Output : Structured SymbolExtractorOutput model
    """
    return Agent(
        name="symbol_extractor",
        model='gemma-4-31b-it',
        description="Extracts archetypal symbols, emotional tone, and setting from a dream.",
        output_schema=SymbolExtractorOutput,  # Enforce structured validation output
        instruction="""
You are a Jungian symbol analyst. Given a dream narrative, extract:
1. symbols   — a list of archetypal objects, figures, or motifs (e.g. "water", "tower", "shadow")
2. emotions  — the dominant emotional tones felt in the dream (e.g. "dread", "wonder")
3. setting   — the primary environment (e.g. "forest", "city", "void")

You will also receive the user's historical symbol list from the MCP dream journal.
Flag any symbol that has appeared in previous dreams by setting "recurring": true.
        """,
    )


agent = make_symbol_extractor()


Writing /kaggle/working/oneiromantia/apps/api/symbol_extractor/agent.py


### `apps/api/pattern_analyst/`

In [7]:
%%writefile {API_DIR}/pattern_analyst/__init__.py
from .agent import make_pattern_analyst

__all__ = ["make_pattern_analyst"]


Writing /kaggle/working/oneiromantia/apps/api/pattern_analyst/__init__.py


In [8]:
%%writefile {API_DIR}/pattern_analyst/agent.py
from google.adk.agents.llm_agent import Agent
from pydantic import BaseModel, Field

class PatternAnalystOutput(BaseModel):
    recurring_clusters: list[list[str]] = Field(description="Groups of symbols that tend to co-occur")
    emotional_arc: str = Field(description="How the user's emotional tone has shifted over recent sessions")
    emerging_themes: list[str] = Field(description="Symbols appearing for the first time or increasing in frequency")

def make_pattern_analyst() -> Agent:
    """
    Sub-agent 2: Pattern Analyst
    Input  : current dream symbols + full symbol history from MCP
    Output : Structured PatternAnalystOutput model
    """
    return Agent(
        name="pattern_analyst",
        model="gemma-4-31b-it",
        description="Identifies recurring themes and symbol clusters across dream sessions.",
        output_schema=PatternAnalystOutput,  # Enforce structured validation output
        instruction="""
You are a dream pattern analyst. You will receive:
- The symbols extracted from today's dream
- A historical frequency table of all symbols the user has ever recorded

Your job is to identify:
1. recurring_clusters — groups of symbols that tend to co-occur
2. emotional_arc      — how the user's emotional tone has shifted over recent sessions
3. emerging_themes    — symbols appearing for the first time or increasing in frequency
        """,
    )


agent = make_pattern_analyst()


Writing /kaggle/working/oneiromantia/apps/api/pattern_analyst/agent.py


### `apps/api/art_generator/`

In [9]:
%%writefile {API_DIR}/art_generator/__init__.py
from .agent import make_art_generator

__all__ = ["make_art_generator"]


Writing /kaggle/working/oneiromantia/apps/api/art_generator/__init__.py


In [10]:
%%writefile {API_DIR}/art_generator/agent.py
from google.adk.agents.llm_agent import Agent
from oneiromantia_art_spec import ART_GENERATOR_INSTRUCTION

def make_art_generator() -> Agent:
    """
    Sub-agent 3: Art Generator
    Input  : art seed JSON from MCP (symbol graph with weights + emotions)
    Output : p5.js sketch code as a string
    """
    return Agent(
        name="art_generator",
        model="gemma-4-31b-it",
        description="Translates a symbol graph into a procedural p5.js sketch.",
        instruction=ART_GENERATOR_INSTRUCTION,
    )


agent = make_art_generator()


Writing /kaggle/working/oneiromantia/apps/api/art_generator/agent.py


### `apps/api/orchestrator/`

In [11]:
%%writefile {API_DIR}/orchestrator/__init__.py
from .agent import make_orchestrator

__all__ = ["make_orchestrator"]


Writing /kaggle/working/oneiromantia/apps/api/orchestrator/__init__.py


In [12]:
%%writefile {API_DIR}/orchestrator/agent.py
# orchestrator/agent.py
import json
import time
import os
import sys
import asyncio
from typing import Any

from google.adk.agents.llm_agent import Agent
from google.adk.workflow import Workflow, START, node
from google.adk.agents.context import Context
from google.adk.events.event import Event
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.genai import types as genai_types

# Resolve the path to the local dream server
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
SERVER_PATH = os.path.abspath(os.path.join(CURRENT_DIR, "../dream_journal_mcp/dream_server.py"))

# Setup the MCP toolset dynamically pointing to the local MCP server using sys.executable
dream_mcp = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=[SERVER_PATH],
        )
    ),
    tool_filter=["save_entry", "get_symbol_history", "get_art_seed", "list_recurring"]
)


def get_content_text(content: Any) -> str:
    """Helper to safely extract string content from Event output or Content structures."""
    if not content:
        return ""
    if isinstance(content, str):
        return content
    if hasattr(content, "parts"):
        parts = []
        for part in content.parts:
            if hasattr(part, "text") and part.text:
                parts.append(part.text)
        return "".join(parts)
    return str(content)


# 1. Intake Node: extract raw dream text and session ID
@node
def intake_node(ctx: Context, node_input: Any) -> Event:
    dream_text = get_content_text(node_input)
    return Event(
        output=dream_text,
        state={
            "raw_text": dream_text,
            "session_id": ctx.session.id
        }
    )


# 2. Context Loader Node: fetch historical symbols and patterns from MCP
@node
async def load_context_node(ctx: Context, node_input: str) -> Event:
    try:
        symbols_contents = await dream_mcp.read_resource("Symbol frequency table")
        symbols_text = symbols_contents[0].text if symbols_contents else "[]"
    except Exception as e:
        print(f"Error reading symbols resource: {e}")
        symbols_text = "[]"

    try:
        patterns_contents = await dream_mcp.read_resource("Recent recurring patterns")
        patterns_text = patterns_contents[0].text if patterns_contents else "{}"
    except Exception as e:
        print(f"Error reading patterns resource: {e}")
        patterns_text = "{}"

    return Event(
        output=node_input,
        state={
            "history_symbols": symbols_text,
            "history_patterns": patterns_text
        }
    )


# 3. Formatter Node: prepare input for symbol_extractor
@node
def prepare_symbol_extractor_input(ctx: Context, node_input: str) -> str:
    history_symbols = ctx.state.get("history_symbols", "[]")
    return f"Dream Narrative:\n{node_input}\n\nHistorical Symbol List (MCP):\n{history_symbols}"


# 4. Processor Node: capture symbol_extractor output & delay 2 seconds (rate limits)
@node
async def process_symbol_extractor_output(ctx: Context, node_input: dict) -> Event:
    # Stagger calls to respect API rate limits
    await asyncio.sleep(2)
    # node_input is a parsed dict because symbol_extractor has an output_schema
    return Event(output=node_input, state={"symbols_output": node_input})


# 5. Formatter Node: prepare input for pattern_analyst
@node
def prepare_pattern_analyst_input(ctx: Context, node_input: dict) -> str:
    history_symbols = ctx.state.get("history_symbols", "[]")
    # node_input is the symbols_output dict from the symbol extractor
    return f"Today's Dream Symbols (JSON):\n{json.dumps(node_input)}\n\nHistorical Symbols Frequency Table (MCP):\n{history_symbols}"


# 6. Processor & MCP Persistence Node: save dream to MCP and prepare art seed
@node
async def save_dream_and_prepare_art(ctx: Context, node_input: dict) -> Event:
    # node_input is a parsed dict from pattern_analyst
    
    # Stagger calls to respect API rate limits
    await asyncio.sleep(2)

    symbols_data = ctx.state.get("symbols_output", {})
    raw_text = ctx.state.get("raw_text", "")
    session_id = ctx.state.get("session_id", "")
    
    # Extract details for the save_entry MCP tool call
    symbols = [s.get("name") if isinstance(s, dict) else str(s) for s in symbols_data.get("symbols", [])]
    emotions = symbols_data.get("emotions", [])
    setting = symbols_data.get("setting", "unknown")

    # Call save_entry tool on MCP
    try:
        await dream_mcp._execute_with_session(
            lambda session: session.call_tool(
                name="save_entry",
                arguments={
                    "raw_text": raw_text,
                    "symbols": symbols,
                    "emotions": emotions,
                    "setting": setting,
                    "session_id": session_id
                }
            ),
            "Failed to save dream entry"
        )
    except Exception as e:
        print(f"Error calling save_entry tool: {e}")

    # Retrieve the structured art seed graph
    try:
        seed_response = await dream_mcp._execute_with_session(
            lambda session: session.call_tool(
                name="get_art_seed",
                arguments={"session_id": session_id}
            ),
            "Failed to get art seed"
        )
        seed_text = "{}"
        if seed_response and hasattr(seed_response, "content"):
            for content_item in seed_response.content:
                if hasattr(content_item, "text") and content_item.text:
                    seed_text = content_item.text
                    break
    except Exception as e:
        print(f"Error calling get_art_seed tool: {e}")
        seed_text = "{}"

    return Event(
        output=seed_text,
        state={
            "patterns_output": node_input,
            "art_seed": seed_text
        }
    )


# 7. Assembly Node: compile everything into the final report structure
@node
def assembly_node(ctx: Context, node_input: Any) -> Event:
    art_text = get_content_text(node_input)
    symbols_data = ctx.state.get("symbols_output", {})
    patterns_data = ctx.state.get("patterns_output", {})

    # Composition matching the required final structure
    report = f"""# Oneiromantia Dream Analysis Report

## ANALYSIS
```json
{json.dumps(symbols_data, indent=2)}
```

## PATTERNS
```json
{json.dumps(patterns_data, indent=2)}
```

## ART_SKETCH
```javascript
{art_text}
```
"""
    # Yield content event for Web UI rendering
    yield Event(content=genai_types.Content(role="model", parts=[genai_types.Part(text=report)]))
    yield Event(output=report)


def make_orchestrator(
    symbol_extractor: Agent,
    pattern_analyst: Agent,
    art_generator: Agent,
) -> Workflow:
    """
    Orchestrator: Compiled Workflow managing intake, context loading, staggered agent
    execution, MCP persistence, and report assembly.
    """
    # Connect the graph edges
    orchestrator = Workflow(
        name="oneiro_orchestrator",
        edges=[
            (START, intake_node),
            (intake_node, load_context_node),
            (load_context_node, prepare_symbol_extractor_input),
            (prepare_symbol_extractor_input, symbol_extractor),
            (symbol_extractor, process_symbol_extractor_output),
            (process_symbol_extractor_output, prepare_pattern_analyst_input),
            (prepare_pattern_analyst_input, pattern_analyst),
            (pattern_analyst, save_dream_and_prepare_art),
            (save_dream_and_prepare_art, art_generator),
            (art_generator, assembly_node)
        ]
    )

    return orchestrator


# --- ADK LOADER BINDING ---
# Import the sub-agents natively so the AgentLoader can discover them and construct the visual graph
from symbol_extractor.agent import agent as symbol_agent
from pattern_analyst.agent import agent as pattern_agent
from art_generator.agent import agent as art_agent

# Instantiate the global agent object that the playground uses to build the graph
agent = make_orchestrator(
    symbol_extractor=symbol_agent,
    pattern_analyst=pattern_agent,
    art_generator=art_agent
)

root_agent = agent

Writing /kaggle/working/oneiromantia/apps/api/orchestrator/agent.py


### `apps/api/dream_journal_mcp/dream_server.py`

In [13]:
%%writefile {API_DIR}/dream_journal_mcp/dream_server.py
import json
import datetime
from mcp.server import Server as MCPServer
from mcp.server.stdio import stdio_server
from mcp import types as mcp_types

# In-memory store for the journal.
_journal: list[dict] = []


def build_mcp_server() -> MCPServer:
    """Construct and wire the Dream Journal MCP server."""
    server = MCPServer("dream-journal")

    # -------------------------------------------------------------------------
    # RESOURCES
    # -------------------------------------------------------------------------

    @server.list_resources()
    async def list_resources() -> list[mcp_types.Resource]:
        return [
            mcp_types.Resource(
                uri="dream://entries",
                name="Dream journal entries",
                description="All dream entries, newest first.",
                mimeType="application/json",
            ),
            mcp_types.Resource(
                uri="dream://symbols/all",
                name="Symbol frequency table",
                description="Every symbol seen across all sessions with counts.",
                mimeType="application/json",
            ),
            mcp_types.Resource(
                uri="dream://patterns/recent",
                name="Recent recurring patterns",
                description="Top recurring themes from the last 30 days.",
                mimeType="application/json",
            ),
        ]

    @server.read_resource()
    async def read_resource(uri: str) -> str:
        uri_str = str(uri)
        # dream://entries — return all entries, newest first
        if uri_str == "dream://entries":
            return json.dumps(list(reversed(_journal)), indent=2)

        # dream://symbols/all — flatten and count every symbol
        if uri_str == "dream://symbols/all":
            freq: dict[str, int] = {}
            for entry in _journal:
                for sym in entry.get("symbols", []):
                    freq[sym] = freq.get(sym, 0) + 1
            ranked = sorted(freq.items(), key=lambda x: x[1], reverse=True)
            return json.dumps([{"symbol": s, "count": c} for s, c in ranked])

        # dream://patterns/recent — symbols from the last 30 days, min count 2
        if uri_str == "dream://patterns/recent":
            cutoff = datetime.datetime.utcnow() - datetime.timedelta(days=30)
            freq: dict[str, int] = {}
            for entry in _journal:
                ts = datetime.datetime.fromisoformat(entry["timestamp"])
                if ts >= cutoff:
                    for sym in entry.get("symbols", []):
                        freq[sym] = freq.get(sym, 0) + 1
            recurring = {s: c for s, c in freq.items() if c >= 2}
            return json.dumps(recurring)

        raise ValueError(f"Unknown resource URI: {uri_str}")

    # -------------------------------------------------------------------------
    # TOOLS
    # -------------------------------------------------------------------------

    @server.list_tools()
    async def list_tools() -> list[mcp_types.Tool]:
        return [
            mcp_types.Tool(
                name="save_entry",
                description="Persist a processed dream entry to the journal.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "raw_text":   {"type": "string"},
                        "symbols":    {"type": "array", "items": {"type": "string"}},
                        "emotions":   {"type": "array", "items": {"type": "string"}},
                        "setting":    {"type": "string"},
                        "session_id": {"type": "string"},
                    },
                    "required": ["raw_text", "symbols", "emotions", "session_id"],
                },
            ),
            mcp_types.Tool(
                name="get_symbol_history",
                description="All past entries containing a given symbol.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "symbol": {"type": "string"},
                    },
                    "required": ["symbol"],
                },
            ),
            mcp_types.Tool(
                name="get_art_seed",
                description=(
                    "Return the structured symbol graph for a session "
                    "as JSON input for the art generator."
                ),
                inputSchema={
                    "type": "object",
                    "properties": {
                        "session_id": {"type": "string"},
                    },
                    "required": ["session_id"],
                },
            ),
            mcp_types.Tool(
                name="list_recurring",
                description="Symbols appearing at least min_count times.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "min_count": {"type": "integer", "default": 2},
                    },
                },
            ),
        ]

    @server.call_tool()
    async def call_tool(name: str, arguments: dict) -> list[mcp_types.TextContent]:
        # save_entry
        if name == "save_entry":
            entry = {
                "session_id": arguments["session_id"],
                "timestamp":  datetime.datetime.utcnow().isoformat(),
                "raw_text":   arguments["raw_text"],
                # NOTE: no raw PII stored beyond what the user typed;
                # in production, encrypt raw_text at rest.
                "symbols":    arguments["symbols"],
                "emotions":   arguments["emotions"],
                "setting":    arguments.get("setting", "unknown"),
            }
            _journal.append(entry)
            return [mcp_types.TextContent(type="text", text=json.dumps({"saved": True, "id": entry["session_id"]}))]

        # get_symbol_history
        if name == "get_symbol_history":
            sym = arguments["symbol"].lower()
            matches = [e for e in _journal if sym in [s.lower() for s in e.get("symbols", [])]]
            return [mcp_types.TextContent(type="text", text=json.dumps(matches))]

        # get_art_seed
        if name == "get_art_seed":
            sid = arguments["session_id"]
            entry = next((e for e in reversed(_journal) if e["session_id"] == sid), None)
            if not entry:
                return [mcp_types.TextContent(type="text", text=json.dumps({"error": "not found"}))]
            # Build a symbol graph: each symbol gets a weight based on
            # how often it has appeared historically.
            freq: dict[str, int] = {}
            for e in _journal:
                for s in e.get("symbols", []):
                    freq[s] = freq.get(s, 0) + 1
            graph = {
                "session_id": sid,
                "emotions":   entry["emotions"],
                "nodes": [
                    {
                        "symbol":    s,
                        "weight":    freq.get(s, 1),       # recurrence score
                        "is_new":    freq.get(s, 0) <= 1,  # first appearance
                    }
                    for s in entry["symbols"]
                ],
            }
            return [mcp_types.TextContent(type="text", text=json.dumps(graph))]

        # list_recurring
        if name == "list_recurring":
            min_count = arguments.get("min_count", 2)
            freq: dict[str, int] = {}
            for e in _journal:
                for s in e.get("symbols", []):
                    freq[s] = freq.get(s, 0) + 1
            result = {s: c for s, c in freq.items() if c >= min_count}
            return [mcp_types.TextContent(type="text", text=json.dumps(result))]

        raise ValueError(f"Unknown tool: {name}")

    return server


async def run_mcp_server():
    """Start the MCP server over stdio (for local ADK connection)."""
    server = build_mcp_server()
    async with stdio_server() as (read_stream, write_stream):
        await server.run(read_stream, write_stream, server.create_initialization_options())


if __name__ == "__main__":
    import asyncio
    asyncio.run(run_mcp_server())

Writing /kaggle/working/oneiromantia/apps/api/dream_journal_mcp/dream_server.py


### `apps/api/server.py`

Included for reference — this is the FastAPI wrapper used when deploying the pipeline as a web service (see `apps/api/Dockerfile`). It is written to disk for completeness but not imported here, since running a blocking `uvicorn` server isn't meaningful inside a notebook cell; the demo below calls the same `build_oneiro_app()` pipeline directly instead.

In [14]:
%%writefile {API_DIR}/server.py
# server.py
import uuid
import logging
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google.genai import types as genai_types

# Import the builder function from main.py
from main import build_oneiro_app

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("oneiro_server")

app = FastAPI(title="Oneiromantia API Server")

# Instantiate the runner globally so it is reused across requests
runner = build_oneiro_app()


class AnalyzeRequest(BaseModel):
    dream: str
    user_id: str = "default-user"


class AnalyzeResponse(BaseModel):
    session_id: str
    report: str
    symbols: dict | None = None
    patterns: dict | None = None
    art_seed: str | None = None


@app.post("/api/analyze", response_model=AnalyzeResponse)
async def analyze_dream(request: AnalyzeRequest):
    if not request.dream.strip():
        raise HTTPException(status_code=400, detail="Dream text cannot be empty.")
    
    session_id = str(uuid.uuid4())
    user_id = request.user_id
    
    logger.info(f"Starting dream analysis for user={user_id}, session={session_id}")
    
    try:
        # 1. Create session
        await runner.session_service.create_session(
            app_name="oneiro",
            user_id=user_id,
            session_id=session_id,
        )
        
        # 2. Run the workflow runner
        final_report = ""
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=genai_types.Content(
                role="user",
                parts=[genai_types.Part(text=request.dream)],
            ),
        ):
            if event.output:
                final_report = event.output
        
        # 3. Retrieve session to get parsed state variables
        session = await runner.session_service.get_session(
            app_name="oneiro",
            user_id=user_id,
            session_id=session_id,
        )
        
        state = session.state if session else {}
        
        return AnalyzeResponse(
            session_id=session_id,
            report=final_report,
            symbols=state.get("symbols_output"),
            patterns=state.get("patterns_output"),
            art_seed=state.get("art_seed")
        )
        
    except Exception as e:
        logger.error(f"Error during dream analysis: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/healthz")
async def health_check():
    return {"status": "ok"}


Writing /kaggle/working/oneiromantia/apps/api/server.py


### `packages/prompts/oneiromantia_art_spec.py`

In [15]:
%%writefile {PACKAGES_DIR}/prompts/oneiromantia_art_spec.py
# =============================================================================
# ONEIROMANTIA — Art Generator: p5.js Output Specification
# =============================================================================
# Drop this instruction string into make_art_generator() in the ADK skeleton,
# replacing the previous instruction value.
#
# The generated sketch:
#   - Renders an animated 3D scene using a manual projection (no WebGL needed)
#   - Uses painter's algorithm (back-to-front sort) for correct depth layering
#   - Evolves continuously via Perlin noise and sinusoidal drift
#   - Maps symbol identity → geometry type
#   - Maps emotion → color palette
#   - Maps symbol weight (recurrence) → density and scale
#   - Supports mouse drag to orbit the camera
# =============================================================================

ART_GENERATOR_INSTRUCTION = """
You are a generative artist. You will receive a symbol graph JSON:
{
  "session_id": "...",
  "emotions": ["wonder", "unease"],
  "nodes": [
    {"symbol": "water",  "weight": 4, "is_new": false},
    {"symbol": "mirror", "weight": 1, "is_new": true}
  ]
}

Generate a COMPLETE, self-contained p5.js sketch. Requirements:

─────────────────────────────────────────────────────────────
1. CANVAS AND PROJECTION
─────────────────────────────────────────────────────────────
- Use p5 in default (2D) mode — NOT WEBGL mode.
- Implement manual 3D projection with two rotation angles:
    camRotX (pitch), camRotY (yaw)
- Project function:
    function project(x, y, z) {
      // rotate around Y
      let rx = x * cos(camRotY) + z * sin(camRotY);
      let rz = -x * sin(camRotY) + z * cos(camRotY);
      // rotate around X
      let ry = y * cos(camRotX) - rz * sin(camRotX);
      let rz2 = y * sin(camRotX) + rz * cos(camRotX);
      let fov = 500;
      let d = fov + rz2;
      if (d < 1) return null;
      let scale = fov / d;
      return { sx: rx * scale, sy: ry * scale, scale, depth: rz2 };
    }
- Center the coordinate system: translate(width/2, height/2) each frame.
- Camera drifts automatically:
    camRotY += speed * 0.0012  (slow continuous yaw)
    camRotX = 0.28 + sin(t * 0.08) * 0.06  (gentle pitch oscillation)
- Mouse drag rotates camera:
    mousePressed → isDragging = true
    mouseDragged → camRotY += (mouseX - pmouseX) * 0.005
                   camRotX -= (mouseY - pmouseY) * 0.004
    mouseReleased → isDragging = false

─────────────────────────────────────────────────────────────
2. CONTINUOUS ANIMATION
─────────────────────────────────────────────────────────────
- Maintain a global float t, incremented each frame by speed * 0.007.
- Use p5.noise() seeded from the session_id hash for all organic motion.
- Every geometry type must use t to drive drift — nothing is static.
- Particles wander via Perlin flow field:
    x += sin(t * 0.3 + phase) * 0.15 * speed
    y += sin(t * 0.2 + phase * 1.3) * 0.12 * speed
    z += cos(t * 0.25 + phase) * 0.12 * speed
  Wrap coordinates at ±280 so particles never disappear.

─────────────────────────────────────────────────────────────
3. DEPTH SORTING (PAINTER'S ALGORITHM)
─────────────────────────────────────────────────────────────
- Collect ALL draw calls into an array: { depth, fn }
- Sort the array by depth descending (furthest first)
- Execute fn() calls in sorted order
- This is mandatory — without it, nearer objects will incorrectly
  appear behind far objects.

─────────────────────────────────────────────────────────────
4. FOG
─────────────────────────────────────────────────────────────
- Compute per-element fog alpha:
    function fogAlpha(depth, base) {
      let fog = constrain((depth + 300) / 600, 0, 1);
      return base * fog;
    }
- Objects deep in the scene fade toward the background color.
- This creates a sense of volumetric space.

─────────────────────────────────────────────────────────────
5. SYMBOL → GEOMETRY MAPPING
─────────────────────────────────────────────────────────────
Scale particle count and ring count by node.weight (recurrence).
Mark is_new nodes with slightly brighter, larger elements.

  "water"  →
    - Sinusoidal wave ribbons: horizontal polylines where y oscillates
      via sin(t * freq + phase + i * 0.4) * amplitude
    - Drifting point cloud (particles) using Perlin flow

  "void"   →
    - Concentric 3D rings: circles in the xz-plane, tilted slightly,
      slowly rotating around Y axis at different speeds
    - Sparse floating particles in a sphere distribution

  "tower"  →
    - Stacked polygons (3–6 sides) at increasing Y heights,
      each rotating independently
    - Vertical particle stream along the Y axis

  "forest" →
    - Recursive branching tree structures using a branch() function:
        function branch(x,y,z, dx,dy,dz, len, gen) {
          if (gen > 4 || len < 4) return;
          // draw segment, then recurse with two child directions
          // sway each branch: x offset += sin(t * 0.4 + phase) * 3 / (gen+1)
        }
    - Leaf-like particle cloud around branch endpoints

  "mirror" →
    - Every point has a reflected counterpart: (x, y, z) → (x, -y, z)
    - Original points in palette color A, reflections in palette color B
      at 50% opacity
    - A single large horizontal ring at y=0 (the mirror plane)

─────────────────────────────────────────────────────────────
6. EMOTION → PALETTE MAPPING
─────────────────────────────────────────────────────────────
Use this exact palette table (RGB arrays):

  wonder:  bg=[6,8,20],   a=[80,200,160],  b=[120,100,220], c=[240,160,60]
  dread:   bg=[4,4,12],   a=[60,40,160],   b=[20,80,80],    c=[100,60,160]
  joy:     bg=[8,6,4],    a=[240,140,60],  b=[220,90,110],  c=[80,200,160]
  anxiety: bg=[10,4,8],   a=[200,60,80],   b=[100,80,180],  c=[160,140,80]
  grief:   bg=[4,8,14],   a=[50,90,160],   b=[60,120,120],  c=[100,100,110]

If multiple emotions are present, blend the two dominant palettes
(linear interpolation, weight = 0.5 each).

─────────────────────────────────────────────────────────────
7. OUTPUT FORMAT
─────────────────────────────────────────────────────────────
Output a SINGLE string containing a complete p5.js sketch with:
  - setup()        : createCanvas(640, 420), colorMode, noiseSeed, buildScene()
  - draw()         : background, advance t, project all elements, depth sort, render
  - buildScene()   : construct all geometry arrays from the symbol graph
  - project(x,y,z) : as specified above
  - fogAlpha(d, b) : as specified above
  - Mouse handlers : mousePressed, mouseDragged, mouseReleased

No markdown, no code fences, no explanation. Plain p5.js code only.
The sketch must run correctly when pasted into editor.p5js.org with no changes.

─────────────────────────────────────────────────────────────
8. EMBEDDING IN THE ONEIROMANTIA UI
─────────────────────────────────────────────────────────────
The sketch will be embedded in an iframe or injected into a page that
already loads the p5.js CDN library. Do not include a <script> tag for p5.
The sketch is the body of a new p5(sketch) instance call — wrap it in:
  const sketch = (p) => { ... };
  new p5(sketch);
Use p.setup, p.draw, p.noise, p.sin, etc. (instance mode).
"""

# =============================================================================
# HOW THE ORCHESTRATOR HANDS OFF TO THE ART GENERATOR
# =============================================================================
#
# After the symbol extractor returns, the orchestrator calls:
#   get_art_seed(session_id)
# on the MCP server, which returns a symbol graph like:
#   {
#     "session_id": "abc-123",
#     "emotions": ["wonder"],
#     "nodes": [
#       {"symbol": "water",  "weight": 4, "is_new": false},
#       {"symbol": "mirror", "weight": 1, "is_new": true}
#     ]
#   }
#
# This JSON is passed verbatim as the art generator's input.
# The art generator returns a p5.js sketch string.
# The orchestrator includes this string in the final output under ART_SKETCH.
#
# In the frontend (Cloud Run web app), the sketch string is injected into
# an iframe via Blob URL:
#
#   const html = `
#     <script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.3/p5.min.js"><\/script>
#     <script>${sketchCode}<\/script>
#   `;
#   const blob = new Blob([html], { type: 'text/html' });
#   iframe.src = URL.createObjectURL(blob);
#
# This keeps the sketch sandboxed and makes it trivially embeddable
# in any frontend without a build step.
# =============================================================================


Writing /kaggle/working/oneiromantia/packages/prompts/oneiromantia_art_spec.py


## 4. Wire up import paths

`apps/api` must be on `sys.path` so `symbol_extractor`, `pattern_analyst`, `art_generator`, `orchestrator`, and `main` import as top-level packages/modules (matching how they're run in the real deployment). `packages/prompts` must be on `sys.path` so `art_generator/agent.py`'s `from oneiromantia_art_spec import ...` resolves — in the real project this happens via a hatchling build `force-include`; here we just add the directory directly.

In [16]:
import sys

for _p in (API_DIR, f"{PACKAGES_DIR}/prompts"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import main
print("Imported main.py from:", main.__file__)


Imported main.py from: /kaggle/working/oneiromantia/apps/api/main.py


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## 5. Wiring self-test (no API key required)

Constructs the full agent graph \u2014 orchestrator, all three sub-agents, and the MCP toolset \u2014 without making any model calls. This always runs, proving the dependencies, materialized source, and imports are all correct independent of whether a usable API key is available.

In [17]:
runner = main.build_oneiro_app()
orchestrator = runner.agent

_seen = []
for _src, _dst in orchestrator.edges:
    for _n in (_src, _dst):
        if _n.name not in _seen:
            _seen.append(_n.name)

print(f"Orchestrator '{orchestrator.name}' constructed with {len(_seen)} nodes:")
for _name in _seen:
    print(" -", _name)

print(
    "\nSelf-test passed: dependencies installed, source materialized, "
    "packages import correctly, and the full agent graph constructs "
    "without errors."
)
if not HAS_KEY:
    print(
        "Skipping the live model demo below (no GOOGLE_API_KEY). "
        "Provide one to see it actually analyze a dream."
    )


Orchestrator 'oneiro_orchestrator' constructed with 11 nodes:
 - __START__
 - intake_node
 - load_context_node
 - prepare_symbol_extractor_input
 - symbol_extractor
 - process_symbol_extractor_output
 - prepare_pattern_analyst_input
 - pattern_analyst
 - save_dream_and_prepare_art
 - art_generator
 - assembly_node

Self-test passed: dependencies installed, source materialized, packages import correctly, and the full agent graph constructs without errors.


## 6. Run the pipeline on a sample dream (live demo)

This exercises the full orchestrator against the real Gemini API: intake -> MCP context load -> `symbol_extractor` -> `pattern_analyst` -> MCP persistence -> `art_generator` -> report assembly — identical to what `POST /api/analyze` does in `server.py`. Skipped if no key is configured; reported (not raised) if the configured key is rejected, so the rest of the notebook still completes.

In [18]:
import uuid
from google.genai import types as genai_types

final_report = ""
state = {}

if not HAS_KEY:
    print("Skipped: no GOOGLE_API_KEY configured. The wiring self-test in "
          "Section 5 already confirmed the pipeline itself is correct.")
else:
    DREAM_TEXT = (
        "I was standing at the edge of a vast black ocean. "
        "The water was completely still, like a mirror. "
        "Behind me was a glass tower that went up beyond the clouds. "
        "I felt a strange mixture of awe and dread \u2014 like I was being watched "
        "by something in the reflection."
    )

    user_id = "kaggle-demo-user"
    session_id = str(uuid.uuid4())

    try:
        await runner.session_service.create_session(
            app_name="oneiro", user_id=user_id, session_id=session_id,
        )

        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=genai_types.Content(
                role="user", parts=[genai_types.Part(text=DREAM_TEXT)]
            ),
        ):
            if event.output:
                final_report = event.output

        session = await runner.session_service.get_session(
            app_name="oneiro", user_id=user_id, session_id=session_id,
        )
        state = session.state if session else {}

        print(final_report)
    except Exception as e:
        print(f"Live demo failed ({type(e).__name__}: {e})\n")
        print(
            "The configured GOOGLE_API_KEY was rejected (invalid, expired, or "
            "out of quota). The wiring self-test in Section 5 already confirmed "
            "the pipeline itself is assembled correctly."
        )


/usr/local/lib/python3.12/dist-packages/google/adk/tools/mcp_tool/mcp_toolset.py:304: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(
Node execution failed with exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 135, in run
    await self._execute_node(ctx, node_input)
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 255, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 269, in _run_node_loop
    async for event in agen:
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_base_node.py", line 217, in run
    async for item in agen:
  File "/usr/local/lib/python3.12/dist-packages/google/adk/agents/llm_agent.py", line 559, in _run_impl
    asyn

Live demo failed (ValidationError: 1 validation error for PatternAnalystOutput
  Invalid JSON: trailing characters at line 6 column 1 [type=json_invalid, input_value=' {\n  "recurring_cluster..., "reflection"]\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid)

The configured GOOGLE_API_KEY was rejected (invalid, expired, or out of quota). The wiring self-test in Section 5 already confirmed the pipeline itself is assembled correctly.


## 7. Inspect the structured outputs

In [19]:
import json

if not state:
    print("No structured output to inspect \u2014 the live demo in Section 6 "
          "did not run or did not complete.")
else:
    print("=== symbols_output ===")
    print(json.dumps(state.get("symbols_output"), indent=2))

    print("\n=== patterns_output ===")
    print(json.dumps(state.get("patterns_output"), indent=2))

    print("\n=== art_seed (symbol graph fed to the art generator) ===")
    print(state.get("art_seed"))


No structured output to inspect — the live demo in Section 6 did not run or did not complete.


## 8. Render the generated p5.js sketch (optional)

The final report embeds the generated sketch under an `ART_SKETCH` code fence. This extracts it and renders it live in an iframe via the p5.js CDN, per the embedding scheme documented in `oneiromantia_art_spec.py`. Requires Internet access enabled for this notebook.

In [20]:
import re
import html as htmlmod
from IPython.display import HTML, display

if not final_report:
    print("No report to render art from \u2014 the live demo in Section 6 "
          "did not run or did not complete.")
else:
    match = re.search(r"## ART_SKETCH\n```javascript\n(.*?)\n```", final_report, re.DOTALL)

    if not match:
        print("No ART_SKETCH block found in the report; skipping render.")
    else:
        sketch_code = match.group(1)
        iframe_doc = f"""<!DOCTYPE html><html><head><meta charset="utf-8"></head>
<body style="margin:0;background:#000;">
<script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.3/p5.min.js"></script>
<script>
{sketch_code}
</script>
</body></html>"""
        escaped = htmlmod.escape(iframe_doc, quote=True)
        display(HTML(
            f'<iframe style="width:660px;height:440px;border:0;background:#000;" '
            f'srcdoc="{escaped}"></iframe>'
        ))


No ART_SKETCH block found in the report; skipping render.
